In [1]:
# ============================================================
# RANDOM FOREST - WEB ATTACK DETECTION 50K
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. CONFIGURATION
RUN_ID = "RF_RUN_001"
ARTIFACT_DIR = "./ML_Artifacts"
RESULT_DIR = "./RandomForest_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

CLASS_NAMES = ["SQLInjection", "XSS", "CommandInjection", "Normal"]

# 2. LOAD ARTIFACTS
print("Loading ML Artifacts...")
X_train = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_train_tfidf.npz"))
X_val = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_val_tfidf.npz"))
y_train = np.load(os.path.join(ARTIFACT_DIR, "y_train.npy"))
y_val = np.load(os.path.join(ARTIFACT_DIR, "y_val.npy"))

print(f"X_train : {X_train.shape} | X_val : {X_val.shape}")

# 3. CREATE & TRAIN MODEL
# n_estimators=200 และ n_jobs=-1 ให้ CPU วิ่งเต็มกำลัง
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nTraining Random Forest...")
start_time = time.time()
model.fit(X_train, y_train)
training_time = time.time() - start_time
print(f"Training Time : {training_time:.2f} seconds ({training_time / 60:.2f} minutes)")

# 4. EVALUATION
start_predict = time.time()
y_pred = model.predict(X_val)
prediction_time = time.time() - start_predict

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average="macro", zero_division=0)
recall = recall_score(y_val, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)

# Command Injection specific
cmd_index = CLASS_NAMES.index("CommandInjection")
cmd_recall = recall_score(y_val, y_pred, labels=[cmd_index], average="macro", zero_division=0)
cmd_f1 = f1_score(y_val, y_pred, labels=[cmd_index], average="macro", zero_division=0)

print("\n" + "=" * 50)
print(f"Accuracy        : {accuracy * 100:.2f}%")
print(f"Macro F1        : {f1 * 100:.2f}%")
print(f"CMDI Recall     : {cmd_recall * 100:.2f}%")
print("=" * 50)

# 5. SAVE RESULTS & CONFUSION MATRIX
result = {
    "run_id": RUN_ID,
    "model": "RandomForest",
    "metrics": {
        "accuracy": float(accuracy),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "command_injection_recall": float(cmd_recall),
        "command_injection_f1": float(cmd_f1)
    },
    "training_time_seconds": float(training_time),
    "prediction_time_seconds": float(prediction_time)
}

with open(os.path.join(RESULT_DIR, f"{RUN_ID}_results.json"), "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)

cm = confusion_matrix(y_val, y_pred)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    os.path.join(RESULT_DIR, f"{RUN_ID}_confusion_matrix.csv"), encoding="utf-8-sig"
)
print("✓ Random Forest results saved successfully")

Loading ML Artifacts...
X_train : (35000, 30000) | X_val : (7500, 30000)

Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:   18.3s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   21.2s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s


Training Time : 21.75 seconds (0.36 minutes)

Accuracy        : 97.57%
Macro F1        : 97.58%
CMDI Recall     : 93.81%
✓ Random Forest results saved successfully


[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.3s finished
